In [ ]:
import threading, time, os, re, subprocess
from datetime import datetime
from pathlib import Path

# === Configuration toggle ===
# HPARAMS = "hparams_jazz_8bars.yaml"        # 8 такт, шаг 4 такта, batch_size 64
HPARAMS = "hparams_jazz_16bars.yaml"     # 16 такт, шаг 4 такта, batch_size 16, dropout 0.2 (paper default)
# HPARAMS = "hparams_jazz_32bars.yaml"     # 32 такт, шаг 4 такта, batch_size 8 (риск OOM, fallback batch_size: 4)
# HPARAMS = "hparams_jazz_16bars_dropout03.yaml"  # 16 такт, dropout 0.3 (ablation: усиленная регуляризация на 422 jazz)
# HPARAMS = "hparams_jazz_16bars_dropout04.yaml"  # 16 такт, dropout 0.4 (ablation: ещё сильнее)
# HPARAMS = "hparams_jazz_16bars_nl4.yaml"  # 16 такт, num_layers 4 (paper-default 8; ablation: capacity downscale)

# SMOKE_MAX_EPOCH = 11    # ~15 min on A100, exercises full pipeline
# SMOKE_MAX_EPOCH = None  # full paper recipe (max_epoch from hparams)
SMOKE_MAX_EPOCH = None

# === Paths ===
# Repo on local SSD (fast, reliable). Reset every Colab session — fresh clone
# avoids ALL FUSE/Drive git issues (lock files, SIGBUS, branch state, etc.).
REPO_ROOT = "/content/repo"
GIT_URL = "https://github.com/kudrmax/CMT-pytorch.git"

# Drive only for persistent training results (checkpoints survive session restarts).
DRIVE_ROOT = "/content/drive/MyDrive/cmt-training"

# diploma2 — для SSoT wjazzd_split.json (cross-model train/eval/test split).
DIPLOMA_REPO_ROOT = "/content/diploma2"
DIPLOMA_GIT_URL = "https://github.com/kudrmax/jazz-generation-research.git"
DIPLOMA_BRANCH = "master"
SPLIT_JSON = f"{DIPLOMA_REPO_ROOT}/pipelines/training-pipeline/wjazzd_split.json"

DATA_ROOT = "jazz/wjazzd/data"
MIDI_DIR_NAME = "midi"
PKL_FILES_ROOT = f"{DATA_ROOT}/pkl_files"
TRAINING_ARTEFACTS_SUBDIR = "training_artefacts"


def run(cmd, **kw):
    """Popen + line-by-line proxy so Colab cell shows child stdout live.

    subprocess.run with inherited stdout doesn't reliably surface child
    output in Colab until child exits. Piping + reading + re-printing
    via parent guarantees real-time display.
    """
    print(f">>> {cmd}", flush=True)
    env = kw.pop("env", None) or os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(
        cmd,
        shell=isinstance(cmd, str),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        text=True,
        **kw,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    return proc


def pkl_restore(pkl_subdir):
    """Restore pkl folder from Drive tar cache. Returns True if restored.

    No-op if local folder already exists or Drive cache is missing.
    Lets `tar -xzf` raise CalledProcessError naturally on corrupted tar —
    we want loud failure, not silent fallback to preprocess.
    """
    local = f"{PKL_FILES_ROOT}/{pkl_subdir}"
    tar_path = f"{DRIVE_ROOT}/pkl_cache/{pkl_subdir}.tar.gz"
    if os.path.isdir(local):
        return False
    if not os.path.exists(tar_path):
        return False
    print(f"==> Restoring {pkl_subdir} from Drive cache...", flush=True)
    os.makedirs(PKL_FILES_ROOT, exist_ok=True)
    run(["tar", "-xzf", tar_path, "-C", f"{PKL_FILES_ROOT}/"])
    return True


def pkl_cache(pkl_subdir):
    """Cache local pkl folder to Drive as tar.gz. Returns True if uploaded.

    No-op if folder is missing or cache already exists. Cache is keyed
    by pkl_dir_name() so different (num_bars, stride_bars, ...) tuples
    get separate archives — no false hits across configs.
    """
    local = f"{PKL_FILES_ROOT}/{pkl_subdir}"
    tar_path = f"{DRIVE_ROOT}/pkl_cache/{pkl_subdir}.tar.gz"
    if not os.path.isdir(local):
        return False
    if os.path.exists(tar_path):
        return False
    print(f"==> Caching {pkl_subdir} to Drive (one-time, future sessions reuse)...", flush=True)
    os.makedirs(os.path.dirname(tar_path), exist_ok=True)
    run(["tar", "-czf", tar_path, "-C", f"{PKL_FILES_ROOT}/", pkl_subdir])
    return True


# 1. Mount Drive (only used for result_root, not for repo)
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_ROOT, exist_ok=True)

# 2. Fresh clone to local SSD — bulletproof. If /content/repo exists from
#    a prior cell run within this session, move it aside (don't delete —
#    keeps any uncommitted state recoverable).
if os.path.isdir(REPO_ROOT):
    backup = f"{REPO_ROOT}.old.{int(time.time())}"
    os.rename(REPO_ROOT, backup)
    print(f"==> Moved old repo to {backup}", flush=True)
run(["git", "clone", "-b", "master", GIT_URL, REPO_ROOT])

# 2.5. Clone diploma2 (shallow) — нужен только wjazzd_split.json для SSoT
#      train/eval/test split (используется в preprocess.py и evaluate_canonical).
if os.path.isdir(DIPLOMA_REPO_ROOT):
    backup = f"{DIPLOMA_REPO_ROOT}.old.{int(time.time())}"
    os.rename(DIPLOMA_REPO_ROOT, backup)
    print(f"==> Moved old diploma2 repo to {backup}", flush=True)
run(["git", "clone", "--depth", "1", "-b", DIPLOMA_BRANCH, DIPLOMA_GIT_URL, DIPLOMA_REPO_ROOT])
assert os.path.exists(SPLIT_JSON), f"split.json missing at {SPLIT_JSON}"

os.chdir(REPO_ROOT)

# 3. Install deps
run(["pip", "install", "-q", "-r", "requirements-data-prep.txt", "-r", "requirements-training.txt"])

# 4. Validate setup
run(["pytest", "-q", "--tb=line"])

# 5. Sanity-check committed input data is present (it came with the clone)
midi_root = f"{DATA_ROOT}/{MIDI_DIR_NAME}"
for path in [f"{DATA_ROOT}/wjazzd.db", midi_root]:
    assert os.path.exists(path), f"committed input missing: {path} — clone failed?"
n_midi = len(os.listdir(midi_root))
assert n_midi >= 400, f"{midi_root} has only {n_midi} dirs, expected ~430"
print(f"==> Input data OK: {n_midi} midi dirs.", flush=True)

# 6. Derive config-specific paths from HPARAMS
import yaml, sys
sys.path.insert(0, ".")
from training.pkl_paths import pkl_dir_name, pkl_preprocess_params
hparams_data = yaml.safe_load(open(HPARAMS))
NUM_BARS, STRIDE_BARS, FPB, PITCH_RANGE = pkl_preprocess_params(hparams_data)
PKL_CKEY_DIR = pkl_dir_name(num_bars=NUM_BARS, stride_bars=STRIDE_BARS,
                             frame_per_bar=FPB, pitch_range=PITCH_RANGE, shift=False)
PKL_CKEY = f"{PKL_FILES_ROOT}/{PKL_CKEY_DIR}"
PKL_12KEYS_DIR = pkl_dir_name(num_bars=NUM_BARS, stride_bars=STRIDE_BARS,
                                frame_per_bar=FPB, pitch_range=PITCH_RANGE, shift=True)
# RESULT_ROOT ключуется по basename hparams-файла (а не только по NUM_BARS):
# baseline `hparams_jazz_16bars.yaml` → `result_16bars` (как было — старые
# артефакты на Drive не перетираются), ablation `hparams_jazz_16bars_dropout03.yaml`
# → `result_16bars_dropout03` (изолированная папка под эксперимент).
RESULT_TAG = Path(HPARAMS).stem.removeprefix("hparams_jazz_")
RESULT_ROOT = f"{DRIVE_ROOT}/result_{RESULT_TAG}"
print(f"==> Config: {HPARAMS} → num_bars={NUM_BARS}, stride_bars={STRIDE_BARS}, batch_size={hparams_data['data_io']['loader']['batch_size']}", flush=True)
print(f"==> Pkl folder (local SSD): {PKL_CKEY}", flush=True)
print(f"==> Result root (Drive): {RESULT_ROOT}", flush=True)

# 7. Run preprocess.py 1-key (Phase 2 input). 12-keys is invoked by orchestrator.
#    Try Drive cache first; if missing, run preprocess + cache. Saves ~3 min on restart.
#
#    --split-json фиксирует SSoT train/eval/test split (test=40 canonical).
#    ВНИМАНИЕ: pkl-кэш keyed только by pkl_dir_name(num_bars, stride_bars, fpb,
#    pitch_range, shift), без учёта split-источника. При первом запуске после
#    введения --split-json удалите вручную старые tar.gz в {DRIVE_ROOT}/pkl_cache/
#    (legacy 70/10/20-split), иначе они будут реюзаться и флаг не вступит в силу.
pkl_restore(PKL_CKEY_DIR)
if not os.path.isdir(PKL_CKEY):
    print("==> Running preprocess.py (1-key, Phase 2 input)...", flush=True)
    run(["python", "preprocess.py",
         "--root_dir", DATA_ROOT, "--midi_dir", MIDI_DIR_NAME,
         "--num_bars", str(NUM_BARS), "--stride_bars", str(STRIDE_BARS),
         "--frame_per_bar", str(FPB), "--pitch_range", str(PITCH_RANGE),
         "--split-json", SPLIT_JSON])
    assert os.path.isdir(PKL_CKEY), f"preprocess did not create {PKL_CKEY}"
pkl_cache(PKL_CKEY_DIR)

# 8. Background ETA monitor — daemon thread, prints every 60s
def monitor_loop():
    while True:
        time.sleep(60)
        try:
            for phase, idx in [("Phase 1", "idx001"), ("Phase 2", "idx002")]:
                log = f"{RESULT_ROOT}/{TRAINING_ARTEFACTS_SUBDIR}/{idx}/log.txt"
                if not os.path.exists(log):
                    continue
                text = open(log).read()
                matches = re.findall(r"(\d{2}-\d{2} \d{2}:\d{2}:\d{2})\..*train (\d+) epoch", text)
                if len(matches) < 2:
                    continue
                ts = [(datetime.strptime(f"2026-{t}", "%Y-%m-%d %H:%M:%S"), int(e)) for t, e in matches]
                deltas = [(ts[i+1][0] - ts[i][0]).total_seconds() for i in range(len(ts) - 1)]
                avg = sum(deltas) / len(deltas)
                last = ts[-1][1]
                derived = f"{RESULT_ROOT}/{TRAINING_ARTEFACTS_SUBDIR}/{idx}/hparams.yaml"
                hp_path = derived if os.path.exists(derived) else HPARAMS
                max_ep = yaml.safe_load(open(hp_path))["experiment"]["max_epoch"]
                eta_min = (max_ep - last) * avg / 60
                print(f"[ETA] {phase}: epoch {last}/{max_ep}, {avg:.0f}s/epoch, ETA {eta_min:.0f} min", flush=True)
        except Exception:
            pass

threading.Thread(target=monitor_loop, daemon=True).start()

# 9. Train. data-root is local SSD (fast). result-root on Drive (persistent).
#    Restore 12-keys pkl from Drive before launching — orchestrator skips internal
#    preprocess if folder + .done marker present. Saves ~50 min on restart.
pkl_restore(PKL_12KEYS_DIR)
train_cmd = ["python", "-m", "training.train",
             "--data-root", DATA_ROOT,
             "--result-root", RESULT_ROOT,
             "--hparams", HPARAMS]
if SMOKE_MAX_EPOCH:
    train_cmd += ["--max-epoch", str(SMOKE_MAX_EPOCH)]
run(train_cmd)
# Cache 12-keys pkl if newly generated by orchestrator (one-time per config).
pkl_cache(PKL_12KEYS_DIR)

# 9.5. Evaluate model_best on canonical test=40 → дописывает final_test_* в summary.json.
#      Это и есть cross-model output-parity с BebopNet (final_test_ppl, final_test_pitch_acc, ...).
#      Сам evaluate_canonical фильтрует pkl по split.json на load-time, так что работает даже
#      если pkl создан со старым legacy 70/10/20-split.
run(["python", "-m", "training.evaluate_canonical",
     "--result-root", RESULT_ROOT,
     "--hparams", HPARAMS,
     "--split-json", SPLIT_JSON])

# 10. Confirm final artefacts (по образцу BebopNet step 10: assert summary.json + print метрик).
run(["ls", "-la", f"{RESULT_ROOT}/best_jazz_model_{NUM_BARS}bars.pth.tar"])
run(f"ls -la {RESULT_ROOT}/*.json")

import json as _json
summary_path = f"{RESULT_ROOT}/summary.json"
assert os.path.exists(summary_path), f"summary.json missing at {summary_path}"
with open(summary_path) as _f:
    _s = _json.load(_f)
assert "final_test_ppl" in _s, "evaluate_canonical did not append final_test_* to summary.json"
_pitch_acc = _s.get("final_test_pitch_acc")
_rhythm_acc = _s.get("final_test_rhythm_acc")
print(
    f"==> Done. final_test_ppl={_s['final_test_ppl']:.3f}, "
    f"pitch_acc={_pitch_acc if _pitch_acc is None else f'{_pitch_acc:.3f}'}, "
    f"rhythm_acc={_rhythm_acc if _rhythm_acc is None else f'{_rhythm_acc:.3f}'}, "
    f"n_files={_s['final_test_n_files']}, n_windows={_s['final_test_n_windows']}",
    flush=True,
)
